In [1]:
# This code gives an overview of the Tourismdata, to help with that I also use the original Paper that used this Dataset to gain more incite 
# This overview is used to then look up, how to transform the Data with best practices for neuronal network use


In [2]:
"""
Tourism dataset overview + complexity diagnostics + reconstruction of the full grouped structure (555 series)

Context from the MinT paper (Wickramasuriya, Athanasopoulos & Hyndman, 2017):
- Metric: "visitor nights" (nights away from home).
- Frequency: monthly.
- Span: Jan 1998 ... Dec 2016 (228 months).
- Grouping: geography (Australia -> 7 states/territories -> 27 zones -> 76 regions)
  and purpose of travel (holiday, visiting friends & relatives (VFR), business, other).
- Total series in the grouped structure (including aggregates): 555 (see Table 7 in the paper).
- Disaggregated series are typically noisier / lower signal-to-noise than aggregates (a common
  reason reconciliation helps in such datasets).

This script:
1) Reads your CSV (wide format).
2) Builds a clean monthly DatetimeIndex.
3) Parses series columns: <REGION_CODE><PURPOSE_CODE> (e.g., "AAAHol").
4) Reshapes to long form and infers hierarchy codes:
   - state_code = region_code[0]
   - zone_code  = region_code[:2]
   - region_code = region_code (3 chars)
5) Computes “complexity” diagnostics: sparsity (zeros), scale heterogeneity, variability,
   trend proxy, and correlation proxy.
6) Reconstructs the full 555-series grouped dataset by aggregating:
   - regions -> zones -> states -> Australia
   - purposes -> total over purposes
"""

from __future__ import annotations

import argparse
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Tuple, List

import numpy as np
import pandas as pd

# Optional plotting (only used if --plots is set)
import matplotlib.pyplot as plt


# -----------------------------
# 0) Paper-aligned metadata
# -----------------------------

PURPOSE_CODE_TO_NAME = {
    "Hol": "holiday",
    "Vis": "visiting_friends_relatives",  # VFR in the paper; CSV uses "Vis"
    "Bus": "business",
    "Oth": "other",
    "Tot": "total_all_purposes",
}

# State/territory codes from the paper's Appendix (Table 6 shows letter labels for states).
# (Names are convenient; aggregation only needs codes.)
STATE_CODE_TO_NAME = {
    "A": "NSW",
    "B": "VIC",
    "C": "QLD",
    "D": "SA",
    "E": "WA",
    "F": "TAS",
    "G": "NT",
}


# -----------------------------
# 1) Loading + parsing
# -----------------------------

MONTH_TO_NUM = {m: i for i, m in enumerate(
    ["January", "February", "March", "April", "May", "June",
     "July", "August", "September", "October", "November", "December"],
    start=1
)}

def build_monthly_index(year_col: pd.Series, month_col: pd.Series) -> pd.DatetimeIndex:
    """
    The provided CSV commonly has Year filled only on the first row of each year (others are NaN).
    We forward-fill Year to build a complete monthly DatetimeIndex.
    """
    year = pd.to_numeric(year_col, errors="coerce").ffill()
    if year.isna().any():
        raise ValueError("Year column contains NaNs even after forward-fill. Check file formatting.")

    year = year.astype(int)

    month_str = month_col.astype(str).str.strip()
    month_num = month_str.map(MONTH_TO_NUM)
    if month_num.isna().any():
        bad = month_str[month_num.isna()].unique().tolist()
        raise ValueError(f"Unrecognized month names: {bad}")

    idx = pd.to_datetime({"year": year, "month": month_num.astype(int), "day": 1})
    idx = pd.DatetimeIndex(idx, name="date")

    # Basic integrity checks: monthly start frequency and uniqueness
    if idx.has_duplicates:
        raise ValueError("Datetime index has duplicates; check Year/Month columns.")
    return idx


def parse_series_column(col: str) -> Tuple[str, str]:
    """
    Expected: <REGION_CODE><PURPOSE_CODE>
      - REGION_CODE: 3 uppercase letters, e.g., AAA, AAB, ...
      - PURPOSE_CODE: Hol, Vis, Bus, Oth
    """
    col = str(col).strip()
    purpose = col[-3:]
    region = col[:-3]

    if purpose not in PURPOSE_CODE_TO_NAME or purpose == "Tot":
        raise ValueError(f"Unexpected purpose suffix in column '{col}'. "
                         f"Expected one of {['Hol','Vis','Bus','Oth']}.")

    if len(region) != 3 or not region.isalpha() or not region.isupper():
        raise ValueError(f"Unexpected region code in column '{col}'. "
                         "Expected 3 uppercase letters (e.g., 'AAA').")

    return region, purpose


def load_wide_csv(csv_path: Path) -> Tuple[pd.DataFrame, pd.DatetimeIndex, Dict]:
    df_raw = pd.read_csv(csv_path)
    required = {"Year", "Month"}
    if not required.issubset(df_raw.columns):
        raise ValueError(f"CSV must contain {required}, but columns are: {df_raw.columns.tolist()}")

    idx = build_monthly_index(df_raw["Year"], df_raw["Month"])
    df_wide = df_raw.drop(columns=["Year", "Month"]).copy()
    df_wide.index = idx

    # Validate series columns
    regions, purposes = [], []
    for c in df_wide.columns:
        r, p = parse_series_column(c)
        regions.append(r)
        purposes.append(p)

    meta = {
        "T": len(df_wide),
        "date_min": df_wide.index.min(),
        "date_max": df_wide.index.max(),
        "n_cols": df_wide.shape[1],
        "regions": sorted(set(regions)),
        "purposes": sorted(set(purposes)),
    }
    return df_wide, idx, meta


def wide_to_bottom_long(df_wide: pd.DataFrame) -> pd.DataFrame:
    """
    Converts wide region×purpose columns to a tidy long dataframe with hierarchy codes.
    """
    # Build a column metadata table
    col_meta = []
    for c in df_wide.columns:
        region, purpose = parse_series_column(c)
        zone = region[:2]      # inferred from code structure used in the paper’s appendix tables
        state = region[0]
        col_meta.append((c, region, zone, state, purpose))

    meta_df = pd.DataFrame(col_meta, columns=["col", "region", "zone", "state", "purpose"])

    # Melt
    long = df_wide.reset_index().melt(id_vars=["date"], var_name="col", value_name="value")
    long = long.merge(meta_df, on="col", how="left").drop(columns=["col"])

    # Add readable names (optional)
    long["purpose_name"] = long["purpose"].map(PURPOSE_CODE_TO_NAME)
    long["state_name"] = long["state"].map(STATE_CODE_TO_NAME).fillna(long["state"])

    return long


# -----------------------------
# 2) Reconstruct the full 555-series grouped structure (Table 7 in paper)
# -----------------------------

def build_grouped_555(bottom_long: pd.DataFrame) -> pd.DataFrame:
    """
    Produces a long dataframe with columns:
      date, geo_level, geo_code, purpose, value
    where purpose includes the 4 purposes plus Tot (sum across purposes).

    Counts match the paper’s Table 7:
      - Australia: 1 geo_code -> 4 purposes + Tot = 5
      - State:     7 geo_codes -> 7*4 + 7 Tot = 35
      - Zone:     27 geo_codes -> 27*4 + 27 Tot = 135
      - Region:   76 geo_codes -> 76*4 + 76 Tot = 380
      Total = 555 series
    """
    def agg(level_name: str, key_col: str) -> pd.DataFrame:
        # per-purpose
        g = (bottom_long
             .groupby(["date", key_col, "purpose"], as_index=False)["value"]
             .sum())
        g = g.rename(columns={key_col: "geo_code"})
        g["geo_level"] = level_name

        # total over purposes
        tot = (g.groupby(["date", "geo_level", "geo_code"], as_index=False)["value"]
               .sum())
        tot["purpose"] = "Tot"

        out = pd.concat([
            g[["date", "geo_level", "geo_code", "purpose", "value"]],
            tot[["date", "geo_level", "geo_code", "purpose", "value"]],
        ], ignore_index=True)
        return out

    region = agg("Region", "region")
    zone   = agg("Zone", "zone")
    state  = agg("State", "state")

    # Australia = sum across all regions (bottom level) for each purpose + total
    aus_p = (bottom_long.groupby(["date", "purpose"], as_index=False)["value"].sum())
    aus_p["geo_level"] = "Australia"
    aus_p["geo_code"] = "AUS"
    aus_tot = (aus_p.groupby(["date", "geo_level", "geo_code"], as_index=False)["value"].sum())
    aus_tot["purpose"] = "Tot"

    australia = pd.concat([
        aus_p[["date", "geo_level", "geo_code", "purpose", "value"]],
        aus_tot[["date", "geo_level", "geo_code", "purpose", "value"]],
    ], ignore_index=True)

    grouped = pd.concat([australia, state, zone, region], ignore_index=True)

    # Sanity check: number of series
    n_series = grouped.drop_duplicates(["geo_level", "geo_code", "purpose"]).shape[0]
    if n_series != 555:
        raise RuntimeError(f"Expected 555 series (paper), got {n_series}. "
                           "Check region/zone/state inference or input columns.")

    return grouped


# -----------------------------
# 3) Complexity diagnostics
# -----------------------------

@dataclass
class ComplexityReport:
    n_timepoints: int
    n_regions: int
    n_purposes: int
    n_bottom_series: int
    zero_share_mean: float
    zero_share_p95: float
    scale_ratio_p95_p05: float
    cv_median: float
    avg_corr_regions: float
    avg_abs_corr_regions: float


def compute_complexity(df_wide: pd.DataFrame, bottom_long: pd.DataFrame) -> ComplexityReport:
    """
    A practical “complexity” snapshot for forecasting:
    - sparsity: fraction of zeros
    - scale heterogeneity: ratio of large vs small series totals
    - variability: coefficient of variation
    - dependence: average pairwise correlation across region totals
    """
    # Sparsity across bottom series (each column is one region×purpose series)
    zero_share = (df_wide.eq(0).sum(axis=0) / len(df_wide)).astype(float)
    zero_mean = float(zero_share.mean())
    zero_p95 = float(zero_share.quantile(0.95))

    # Scale heterogeneity: total volume per series
    series_totals = df_wide.sum(axis=0)
    q05 = float(series_totals.quantile(0.05))
    q95 = float(series_totals.quantile(0.95))
    scale_ratio = float(q95 / q05) if q05 != 0 else np.inf

    # Variability proxy: coefficient of variation per series
    series_mean = df_wide.mean(axis=0).replace(0, np.nan)
    series_std = df_wide.std(axis=0)
    cv = (series_std / series_mean).replace([np.inf, -np.inf], np.nan).dropna()
    cv_median = float(cv.median()) if len(cv) else float("nan")

    # Correlation proxy across region totals (sum across purposes within region)
    region_totals = (bottom_long[bottom_long["purpose"] != "Tot"]
                     .groupby(["date", "region"], as_index=False)["value"]
                     .sum()
                     .pivot(index="date", columns="region", values="value"))
    corr = region_totals.corr()
    # average off-diagonal
    mask = ~np.eye(corr.shape[0], dtype=bool)
    avg_corr = float(corr.where(mask).stack().mean())
    avg_abs_corr = float(corr.abs().where(mask).stack().mean())

    # Counts
    regions = bottom_long["region"].nunique()
    purposes = bottom_long["purpose"].nunique()

    return ComplexityReport(
        n_timepoints=len(df_wide),
        n_regions=int(regions),
        n_purposes=int(purposes),
        n_bottom_series=int(df_wide.shape[1]),
        zero_share_mean=zero_mean,
        zero_share_p95=zero_p95,
        scale_ratio_p95_p05=scale_ratio,
        cv_median=cv_median,
        avg_corr_regions=avg_corr,
        avg_abs_corr_regions=avg_abs_corr,
    )


def print_overview(df_wide: pd.DataFrame, meta: Dict, bottom_long: pd.DataFrame, grouped_555: pd.DataFrame) -> None:
    # Basic structure
    print("\n=== BASIC STRUCTURE ===")
    print(f"Time span: {meta['date_min'].date()} .. {meta['date_max'].date()}  (T={meta['T']} monthly obs)")
    print(f"Wide shape: {df_wide.shape}  [rows=time, cols=region×purpose series]")
    print(f"Regions: {len(meta['regions'])}  Purposes: {len(meta['purposes'])}")
    print(f"Bottom-level series in CSV: {df_wide.shape[1]}  (= regions×purposes)")

    # Missingness
    print("\n=== MISSINGNESS ===")
    na_total = int(df_wide.isna().sum().sum())
    print(f"Total NaNs in series matrix: {na_total}")
    if na_total:
        na_by_col = df_wide.isna().mean(axis=0).sort_values(ascending=False).head(10)
        print("Top-10 columns by NaN rate:")
        print(na_by_col)

    # Hierarchy inferred from codes
    print("\n=== INFERRED HIERARCHY SHAPE (from region code) ===")
    n_states = bottom_long["state"].nunique()
    n_zones = bottom_long["zone"].nunique()
    n_regions = bottom_long["region"].nunique()
    print(f"States: {n_states}  Zones: {n_zones}  Regions: {n_regions}")
    print("States found:", sorted(bottom_long["state"].unique().tolist()))

    # Check grouped-structure counts
    print("\n=== RECONSTRUCTED GROUPED DATASET (paper's Table 7 target) ===")
    series_counts = grouped_555.drop_duplicates(["geo_level", "geo_code", "purpose"]).groupby("geo_level").size()
    print(series_counts)
    print(f"Total series: {int(series_counts.sum())} (should be 555)")

    # Quick “what does the data represent?”
    print("\n=== SEMANTICS OF VALUES (practical) ===")
    # Australia total over purposes
    aus_tot = (grouped_555[(grouped_555["geo_level"] == "Australia") & (grouped_555["purpose"] == "Tot")]
               .set_index("date")["value"])
    desc = aus_tot.describe()
    print("Australia total (all purposes) summary stats:")
    print(desc)
    print("Note: In the paper, plots are shown in 'millions' of visitor nights;")
    print("your file values may therefore be in 'thousands' (so 48,000 ~ 48 million), depending on source scaling.")

    # Top series by total volume
    print("\n=== TOP SERIES BY TOTAL VOLUME (bottom-level region×purpose) ===")
    totals = df_wide.sum(axis=0).sort_values(ascending=False).head(10)
    print(totals)


def make_plots(grouped_555: pd.DataFrame, out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)

    # Plot Australia by purpose
    aus = grouped_555[(grouped_555["geo_level"] == "Australia") & (grouped_555["purpose"].isin(["Hol", "Vis", "Bus", "Oth", "Tot"]))]
    pivot = aus.pivot(index="date", columns="purpose", values="value").sort_index()

    plt.figure()
    for p in pivot.columns:
        plt.plot(pivot.index, pivot[p], label=p)
    plt.title("Australia visitor nights by purpose (and total)")
    plt.xlabel("Date")
    plt.ylabel("Value (units as in file)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dir / "australia_by_purpose.png", dpi=150)
    plt.close()

    # Distribution of bottom-series totals
    plt.figure()
    totals = pivot = None  # avoid accidental reuse
    plt.hist([], bins=30)  # placeholder reset (safe even if removed)

    # Recreate totals from grouped_555 for regions×purposes only (excluding Tot)
    # (Not required; you can also compute from df_wide directly and pass in.)
    plt.close()  # keep plots minimal if you extend this


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", type=Path, default=Path("TourismData_Rob_J_Hydenan.csv"))
    ap.add_argument("--out_dir", type=Path, default=Path("tourism_overview_outputs"))
    ap.add_argument("--export_grouped_csv", action="store_true")
    ap.add_argument("--plots", action="store_true")
    args, unknown = ap.parse_known_args()  # <-- change here

    # Optional: show what got ignored (handy in notebooks)
    # if unknown:
    #     print("Ignored unknown args:", unknown)

    df_wide, idx, meta = load_wide_csv(args.csv)
    bottom_long = wide_to_bottom_long(df_wide)
    grouped_555 = build_grouped_555(bottom_long)

    report = compute_complexity(df_wide, bottom_long)
    print_overview(df_wide, meta, bottom_long, grouped_555)

    print("\n=== COMPLEXITY DIAGNOSTICS (practical) ===")
    print(f"- Mean zero share across bottom series: {report.zero_share_mean:.3f}")
    print(f"- 95th percentile zero share: {report.zero_share_p95:.3f}")
    print(f"- Scale ratio (p95 total / p05 total): {report.scale_ratio_p95_p05:.2f}")
    print(f"- Median coefficient of variation (std/mean): {report.cv_median:.2f}")
    print(f"- Avg pairwise correlation across region totals: {report.avg_corr_regions:.3f}")
    print(f"- Avg |correlation| across region totals: {report.avg_abs_corr_regions:.3f}")

    if args.export_grouped_csv:
        args.out_dir.mkdir(parents=True, exist_ok=True)
        out_path = args.out_dir / "tourism_grouped_555_long.csv"
        grouped_555.to_csv(out_path, index=False)
        print(f"\nExported: {out_path}")

    if args.plots:
        make_plots(grouped_555, args.out_dir)
        print(f"\nPlots saved to: {args.out_dir.resolve()}")



if __name__ == "__main__":
    main()



=== BASIC STRUCTURE ===
Time span: 1998-01-01 .. 2016-12-01  (T=228 monthly obs)
Wide shape: (228, 304)  [rows=time, cols=region×purpose series]
Regions: 76  Purposes: 4
Bottom-level series in CSV: 304  (= regions×purposes)

=== MISSINGNESS ===
Total NaNs in series matrix: 0

=== INFERRED HIERARCHY SHAPE (from region code) ===
States: 7  Zones: 27  Regions: 76
States found: ['A', 'B', 'C', 'D', 'E', 'F', 'G']

=== RECONSTRUCTED GROUPED DATASET (paper's Table 7 target) ===
geo_level
Australia      5
Region       380
State         35
Zone         135
dtype: int64
Total series: 555 (should be 555)

=== SEMANTICS OF VALUES (practical) ===
Australia total (all purposes) summary stats:
count      228.000000
mean     23636.266779
std       6691.496608
min      15534.870987
25%      19761.710958
50%      22084.025597
75%      24854.834393
max      48008.859101
Name: value, dtype: float64
Note: In the paper, plots are shown in 'millions' of visitor nights;
your file values may therefore be in 

In [ ]:
# No cleaning needed due to non nans or missing information, all are non negative
#Shape: 228 rows × 306 columns = Year, Month + 304 series (76 regions × 4 purposes).
# Zeros are common: overall ≈ 18% of all entries are exactly 0